In [ ]:
# import libraries
import re
import os
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

In [ ]:
# functions
# Text Cleaning
def clean_text(t):
    # URL removal
    t = re.sub(r"http\S+", "", t)
    # special character removal
    t = re.sub(r"[^a-zA-Z\s]", "", t)
    # whitespace normalization
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Tokenization and Normalization
def tokenize_normalize(t):
    # For logistinc, SVM and RF, apply tokenization, lowercasing and stopword handling
    t = t.lower()
    tokens = t.split()
    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

# Train/Validation/Test Split
def split_data(df):
    train, tmp = train_test_split(df, test_size=0.3, random_state=42)
    val, test = train_test_split(tmp, test_size=0.5, random_state=42)
    return train, val, test

In [ ]:
# reddit dataset
# mental health
reddit_mh_path = "data/raw/reddit/mental_health"
reddit_mh_files = [f for f in os.listdir(reddit_mh_path)]
mh_train = []
mh_val = []
mh_test = []

for f in reddit_mh_files:
    file_path = os.path.join(reddit_mh_path, f)
    df = pd.read_csv(file_path)
    # pick up useful keys
    df = df[['subreddit', 'post']]
    # add label: mental health = 1, control = 0
    df['label'] = 1
    df['cleaned_text'] = df['post'].apply(clean_text)
    df["tfidf_text"] = df["cleaned_text"].apply(lambda x: tokenize_normalize(x, model="tfidf"))
    # split each subreddit(category) into 70% train, 15% validation and 15% test
    train, val, test = split_data(df)
    mh_train.append(train)
    mh_val.append(val)
    mh_test.append(test)


# control
reddit_c_path = "data/raw/reddit/control"
reddit_c_files = [f for f in os.listdir(reddit_c_path)]
c_train = []
c_val = []
c_test = []

for f in reddit_c_files:
    file_path = os.path.join(reddit_c_path, f)
    df = pd.read_csv(file_path)
    # pick up useful keys
    df = df[['subreddit', 'post']]
    # add label: mental health = 1, control = 0
    df['label'] = 0
    df['cleaned_text'] = df['post'].apply(clean_text)
    df["tfidf_text"] = df["cleaned_text"].apply(lambda x: tokenize_normalize(x, model="tfidf"))
    # split each subreddit(category) into 70% train, 15% validation and 15% test
    train, val, test = split_data(df)
    c_train.append(train)
    c_val.append(val)
    c_test.append(test)

# combine mental health and control data
df_reddit_train = pd.concat(mh_train+ c_train, ignore_index=True)
df_reddit_val = pd.concat(mh_val + c_val, ignore_index=True)
df_reddit_test = pd.concat(mh_test + c_val, ignore_index=True)


# fit vectorizer to reddit train data
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(df_reddit_train["tfidf_text"])
# save vectorizer
# saved in "reddit" folder, but used the same vectorizer for twitter dataset
joblib.dump(vectorizer, "data/preprocessed/reddit/vectorizer/reddit_vectorizer_fixed.pkl")

# training data
df_tfidf_train = pd.DataFrame(X_train.toarray())
df_tfidf_train["subreddit"] = df_reddit_train["subreddit"]
df_tfidf_train["label"] = df_reddit_train["label"]
df_tfidf_train.to_csv("data/preprocessed/reddit/reddit_tfidf_train_fixed.csv", index=False)

# val data
X_val = vectorizer.transform(df_reddit_val["tfidf_text"])
df_tfidf_val = pd.DataFrame(X_val.toarray())
df_tfidf_val["subreddit"] = df_reddit_val["subreddit"]
df_tfidf_val["label"] = df_reddit_val["label"]
df_tfidf_val.to_csv("data/preprocessed/reddit/reddit_tfidf_val_fixed.csv", index=False)

# test data
X_test = vectorizer.transform(df_reddit_test["tfidf_text"])
df_tfidf_test = pd.DataFrame(X_test.toarray())
df_tfidf_test["subreddit"] = df_reddit_test["subreddit"]
df_tfidf_test["label"] = df_reddit_test["label"]
df_tfidf_test.to_csv("data/preprocessed/reddit/reddit_tfidf_test_fixed.csv", index=False)

In [ ]:
# twitter dataset
twitter_path = "data/raw/twitter/Mental-Health-Twitter.csv"
df = pd.read_csv(twitter_path)
# pick up useful keys
df = df[["post_text", "label"]]
# preprocessing
df["cleaned_text"] = df["post_text"].apply(clean_text)
df["tfidf_text"] = df["cleaned_text"].apply(lambda x: tokenize_normalize(x, model="tfidf"))

# for logistic, SVM and RF
X_tfidf = vectorizer.transform(df["tfidf_text"])
df_tfidf = pd.DataFrame(X_tfidf.toarray())
df_tfidf["label"] = df["label"]
df_tfidf.to_csv("data/preprocessed/twitter/twitter_tfidf.csv", index=False)